In [ ]:
# === 1. 라이브러리 / 시드 ===
import os
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import log_loss
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# === 2. 데이터 경로 / 컬럼명 해석 (env override 지원) ===
def resolve_data_dir() -> Path:
    env = os.environ.get("ASSIGNMENT3_DATA_DIR")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd()
    for cand in [here / "data", here.parent / "data", here]:
        if cand.exists():
            return cand.resolve()
    return here.resolve()

def resolve_csv(data_dir: Path, exact: str, keyword: str) -> Path:
    direct = data_dir / exact
    if direct.exists():
        return direct.resolve()
    matches = sorted(p for p in data_dir.glob("*.csv") if keyword.lower() in p.name.lower())
    if not matches:
        raise FileNotFoundError(f"Could not find {exact!r} or any csv containing {keyword!r} under {data_dir}")
    return matches[0].resolve()

DATA_DIR = resolve_data_dir()
TRAIN_FILENAME = os.environ.get("ASSIGNMENT3_TRAIN", "train_set.csv")
TEST_FILENAME = os.environ.get("ASSIGNMENT3_TEST", "test_set.csv")
TRAIN_PATH = resolve_csv(DATA_DIR, TRAIN_FILENAME, "train")
TEST_PATH = resolve_csv(DATA_DIR, TEST_FILENAME, "test")
ID_COL, TARGET_COL = "ID", "Delay"
POSITIVE_LABEL, NEGATIVE_LABEL = "Delayed", "Not_Delayed"

In [ ]:
# === 21. 모델링 — CONFIG (경로/컬럼/인코딩/CV/임계값/XGBoost 파라미터 일괄 정의) ===
@dataclass
class Config:
    data_dir: Path = field(default_factory=resolve_data_dir)
    train_filename: str = os.environ.get("ASSIGNMENT3_TRAIN", "train_set.csv")
    test_filename: str = os.environ.get("ASSIGNMENT3_TEST", "test_set.csv")
    submission_dir: Path = field(default_factory=lambda: Path.cwd() / "submissions")
    submission_filename: str = os.environ.get("ASSIGNMENT3_SUBMISSION", "ML_HW3_test.csv")

    id_col: str = "ID"
    target_col: str = "Delay"
    positive_label: str = "Delayed"
    negative_label: str = "Not_Delayed"

    drop_cols: tuple = ("Cancelled", "Diverted", "Airline")
    time_cols: tuple = ("Estimated_Departure_Time", "Estimated_Arrival_Time")
    missing_flag_cols: tuple = ("Estimated_Departure_Time", "Estimated_Arrival_Time",
                                "Origin_State", "Destination_State",
                                "Carrier_Code(IATA)", "Carrier_ID(DOT)", "Tail_Number")

    onehot_cols: tuple = ("Carrier_Group", "Origin_State", "Destination_State",
                          "Carrier_Code(IATA)", "Carrier_ID(DOT)",
                          "Origin_Airport", "Destination_Airport",
                          "Dep_TimeBucket", "Arr_TimeBucket")
    freq_only_cols: tuple = ("Tail_Number", "Route", "Carrier_Route")
    onehot_top_k: int = 50
    onehot_other_label: str = "__other__"

    time_buckets: tuple = (("새벽", 0, 5), ("오전", 6, 11), ("오후", 12, 17), ("저녁", 18, 23))
    bucket_missing_label: str = "__missing__"

    n_splits: int = 5
    pseudo_threshold: float = 0.9

    # XGBoost 베이스라인 파라미터 (튜닝에서 이걸 베이스로 override) — Log-loss 채점 기준
    baseline_params: dict = field(default_factory=lambda: {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "learning_rate": 0.1, "max_depth": 6, "min_child_weight": 1.0,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "reg_alpha": 0.0, "reg_lambda": 1.0,
        "random_state": RANDOM_STATE, "n_jobs": -1,
    })
    baseline_num_boost_round: int = 2000
    early_stopping_rounds: int = 50

CONFIG = Config()

In [ ]:
# === 22. 모델링 — raw_train/raw_test 재로드 + test_ids 보관 ===
raw_train = pd.read_csv(TRAIN_PATH)
raw_test = pd.read_csv(TEST_PATH)
test_ids = raw_test[CONFIG.id_col].copy()

In [ ]:
# === 23. 전처리 — drop_cols + ID 제거 ===
def drop_columns(df, cols):
    return df.drop(columns=[c for c in cols if c in df.columns], errors="ignore")

train_df = drop_columns(raw_train, CONFIG.drop_cols).drop(columns=[CONFIG.id_col], errors="ignore").copy()
test_df = drop_columns(raw_test, CONFIG.drop_cols).drop(columns=[CONFIG.id_col], errors="ignore").copy()

In [ ]:
# === 24. 전처리 — 시각 → hour + 4분할 버킷 + Duration ===
def extract_hour(v):
    if pd.isna(v): return np.nan
    v = int(v); h, m = divmod(v, 100)
    return float(h) if (0 <= h <= 23 and 0 <= m <= 59) else np.nan

def bucketize(h, buckets):
    if pd.isna(h): return CONFIG.bucket_missing_label
    for name, lo, hi in buckets:
        if lo <= int(h) <= hi: return name
    return CONFIG.bucket_missing_label

def add_time_features(df):
    out = df.copy()
    dep_col, arr_col = CONFIG.time_cols
    dep = out[dep_col].apply(extract_hour); arr = out[arr_col].apply(extract_hour)
    out["Dep_Hour"], out["Arr_Hour"] = dep, arr
    out["Dep_TimeBucket"] = dep.apply(lambda h: bucketize(h, CONFIG.time_buckets))
    out["Arr_TimeBucket"] = arr.apply(lambda h: bucketize(h, CONFIG.time_buckets))
    out["Estimated_Duration_min"] = ((arr - dep) % 24) * 60
    return out.drop(columns=list(CONFIG.time_cols), errors="ignore")

train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

In [ ]:
# === 25. 전처리 — *_is_missing 플래그 생성 ===
def add_missing_flags(df, cols):
    out = df.copy()
    for c in cols:
        src = {"Estimated_Departure_Time": "Dep_Hour", "Estimated_Arrival_Time": "Arr_Hour"}.get(c, c)
        if src in out.columns:
            out[f"{c}_is_missing"] = out[src].isna().astype(int)
    return out

train_df = add_missing_flags(train_df, CONFIG.missing_flag_cols)
test_df = add_missing_flags(test_df, CONFIG.missing_flag_cols)

In [ ]:
# === 26. 전처리 — 계절 플래그 + Route + Carrier_Route ===
SUMMER_MONTHS, FALL_LOW_MONTHS = {6, 7, 8}, {9, 10}

def add_derived(df):
    out = df.copy()
    out["Is_Summer"] = out["Month"].isin(SUMMER_MONTHS).astype(int)
    out["Is_FallLow"] = out["Month"].isin(FALL_LOW_MONTHS).astype(int)
    o = out["Origin_Airport"].fillna(CONFIG.bucket_missing_label).astype(str)
    d = out["Destination_Airport"].fillna(CONFIG.bucket_missing_label).astype(str)
    out["Route"] = o + "_" + d
    c = out["Carrier_ID(DOT)"].fillna(-1).astype(int).astype(str)
    out["Carrier_Route"] = c + "_" + out["Route"]
    return out

train_df = add_derived(train_df); test_df = add_derived(test_df)

In [ ]:
# === 27. 전처리 — 타깃 0/1 매핑 + labeled / unlabeled / test 분리 ===
LABEL_MAP = {CONFIG.negative_label: 0, CONFIG.positive_label: 1}
y_full = train_df[CONFIG.target_col].map(LABEL_MAP)
X_full = train_df.drop(columns=[CONFIG.target_col])

labeled_mask = y_full.notna()
labeled_df = X_full.loc[labeled_mask].reset_index(drop=True)
y_labeled = y_full.loc[labeled_mask].astype(int).reset_index(drop=True)
unlabeled_df = X_full.loc[~labeled_mask].reset_index(drop=True)
test_df_X = test_df.copy()

In [ ]:
# === 28. 분석 — 항공사 효과 통계 검증 (chi2 + Bonferroni) ===
AIRLINE_COL_CANDIDATES = ("Airline", "Carrier_ID(DOT)", "Carrier_Code(IATA)")
MIN_SAMPLES_PER_AIRLINE = 100
SIGNIFICANCE_ALPHA = 0.05

def _pick(df, cands):
    for c in cands:
        if c in df.columns: return c
    raise KeyError(cands)

def _wilson_ci(p, n, a=SIGNIFICANCE_ALPHA):
    if n == 0: return (np.nan, np.nan)
    z = stats.norm.ppf(1 - a / 2)
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half

def _two_prop_z(d_a, n_a, d_b, n_b):
    if not (n_a and n_b): return 0.0, 1.0
    p_pool = (d_a + d_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    if se == 0: return 0.0, 1.0
    z = (d_a / n_a - d_b / n_b) / se
    return float(z), float(2 * (1 - stats.norm.cdf(abs(z))))

airline_col = _pick(labeled_df, AIRLINE_COL_CANDIDATES)
air = labeled_df[airline_col].fillna(CONFIG.bucket_missing_label).astype(str)
y = y_labeled.astype(int); base_rate = float(y.mean())

grp = pd.DataFrame({"n": y.groupby(air).size(), "delayed": y.groupby(air).sum()})
grp["rate"] = grp["delayed"] / grp["n"]
grp[["ci_lo", "ci_hi"]] = grp.apply(lambda r: pd.Series(_wilson_ci(r["rate"], int(r["n"]))), axis=1)
grp["diff_vs_base"] = grp["rate"] - base_rate
big = grp[grp["n"] >= MIN_SAMPLES_PER_AIRLINE].sort_values("rate")

ct = pd.crosstab(air, y)
chi2, p_chi2, dof, _ = stats.chi2_contingency(ct)
cramer_v = float(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))))

total_d, total_n = int(y.sum()), len(y)
bonf = SIGNIFICANCE_ALPHA / max(len(big), 1)
rows = []
for name, r in big.iterrows():
    n_a, d_a = int(r["n"]), int(r["delayed"])
    z, p = _two_prop_z(d_a, n_a, total_d - d_a, total_n - n_a)
    rows.append({"airline": name, "n": n_a, "rate": d_a / n_a,
                 "z": z, "p": p, "sig_bonf": p < bonf})
pair_df = pd.DataFrame(rows).sort_values("z")
SIG_AIRLINES = set(pair_df.loc[pair_df["sig_bonf"], "airline"].astype(str).tolist())

In [ ]:
# === 29. 전처리 — Carrier_Group (통과 항공사만 ID 유지, 나머지는 OTHER) ===
OTHER_CARRIER_LABEL = "__other_carrier__"
CARRIER_GROUP_COL = "Carrier_Group"

def assign_carrier_group(df, sig_set, source_col, missing_label):
    s = df[source_col].fillna(missing_label).astype(str)
    return s.where(s.isin(sig_set), OTHER_CARRIER_LABEL)

for df_ in (labeled_df, unlabeled_df, test_df_X):
    df_[CARRIER_GROUP_COL] = assign_carrier_group(df_, SIG_AIRLINES, airline_col, CONFIG.bucket_missing_label)

In [ ]:
# === 30. 인코더 정의 — TopK OneHot + Frequency (y 미사용 → 누수 0) ===
class FrequencyEncoder:
    def __init__(self): self.freq_ = {}
    def fit(self, s):
        self.freq_ = s.fillna(CONFIG.bucket_missing_label).astype(str).value_counts().to_dict()
        return self
    def transform(self, s):
        return (s.fillna(CONFIG.bucket_missing_label).astype(str)
                  .map(self.freq_).fillna(0).astype(float))

class TopKOneHotEncoder:
    # 카디 > top_k 면 상위 K개만 유지, 나머지는 OTHER 단일 컬럼
    def __init__(self, top_k, other_label):
        self.top_k, self.other_label = top_k, other_label
        self.kept_, self.categories_ = set(), []
    def fit(self, s):
        s = s.fillna(CONFIG.bucket_missing_label).astype(str)
        counts = s.value_counts()
        if len(counts) > self.top_k:
            self.kept_ = set(counts.head(self.top_k).index)
            self.categories_ = list(counts.head(self.top_k).index) + [self.other_label]
        else:
            self.kept_ = set(counts.index); self.categories_ = list(counts.index)
        return self
    def transform(self, s):
        s = s.fillna(CONFIG.bucket_missing_label).astype(str)
        m = s.where(s.isin(self.kept_), self.other_label)
        return pd.DataFrame({c: (m == c).astype(np.int8).values for c in self.categories_}, index=s.index)

def encode_train_valid(X_tr, y_tr, X_va):
    # One-Hot + Frequency 인코딩 (y_tr 호환용 미사용). fit은 X_tr만.
    X_tr_enc = X_tr.reset_index(drop=True).copy()
    X_va_enc = X_va.reset_index(drop=True).copy()
    encoders = {"onehot": {}, "freq": {}}
    for col in CONFIG.onehot_cols:
        if col not in X_tr_enc.columns: continue
        enc = TopKOneHotEncoder(CONFIG.onehot_top_k, CONFIG.onehot_other_label).fit(X_tr_enc[col])
        tr_oh = enc.transform(X_tr_enc[col]).reset_index(drop=True)
        va_oh = enc.transform(X_va_enc[col]).reset_index(drop=True)
        tr_oh.columns = [f"{col}__{c}" for c in tr_oh.columns]
        va_oh.columns = [f"{col}__{c}" for c in va_oh.columns]
        X_tr_enc = pd.concat([X_tr_enc.drop(columns=[col]), tr_oh], axis=1)
        X_va_enc = pd.concat([X_va_enc.drop(columns=[col]), va_oh], axis=1)
        encoders["onehot"][col] = enc
    for col in CONFIG.freq_only_cols:
        if col not in X_tr_enc.columns: continue
        fe = FrequencyEncoder().fit(X_tr_enc[col])
        X_tr_enc[f"{col}__freq"] = fe.transform(X_tr_enc[col]).values
        X_va_enc[f"{col}__freq"] = fe.transform(X_va_enc[col]).values
        X_tr_enc = X_tr_enc.drop(columns=[col]); X_va_enc = X_va_enc.drop(columns=[col])
        encoders["freq"][col] = fe
    return X_tr_enc, X_va_enc, encoders

In [ ]:
# === 33. Stratified 5-Fold CV (과제 필수) — OOF log-loss ===
def run_cv(params, X, y, n_splits=CONFIG.n_splits,
           num_boost_round=None, early_stopping_rounds=None, verbose=False):
    num_boost_round = num_boost_round or CONFIG.baseline_num_boost_round
    early_stopping_rounds = early_stopping_rounds or CONFIG.early_stopping_rounds
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=float)
    fold_scores, best_iters = [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx].reset_index(drop=True); X_va = X.iloc[va_idx].reset_index(drop=True)
        y_tr = y.iloc[tr_idx].reset_index(drop=True); y_va = y.iloc[va_idx].reset_index(drop=True)
        X_tr_enc, X_va_enc, _ = encode_train_valid(X_tr, y_tr, X_va)
        dtr, dva = xgb.DMatrix(X_tr_enc, label=y_tr), xgb.DMatrix(X_va_enc, label=y_va)
        b = xgb.train(params, dtr, num_boost_round, evals=[(dva, "valid")],
                      early_stopping_rounds=early_stopping_rounds, verbose_eval=200 if verbose else False)
        p = b.predict(dva, iteration_range=(0, b.best_iteration + 1))
        oof[va_idx] = p
        fold_scores.append(log_loss(y_va, p)); best_iters.append(b.best_iteration)
    overall = log_loss(y, oof)
    return {"oof": oof, "fold_scores": fold_scores, "overall": overall,
            "mean_best_iter": int(np.mean(best_iters))}

cv_baseline = run_cv(CONFIG.baseline_params, labeled_df, y_labeled)
cv_baseline["overall"]

In [ ]:
# === 34. 하이퍼파라미터 튜닝 (4-grid, CV log-loss 기준) ===
PARAM_GRID = [
    {"learning_rate": 0.05, "max_depth": 6, "min_child_weight": 1.0, "subsample": 0.9,  "colsample_bytree": 0.9,  "reg_lambda": 1.0},
    {"learning_rate": 0.05, "max_depth": 8, "min_child_weight": 5.0, "subsample": 0.8,  "colsample_bytree": 0.8,  "reg_lambda": 1.0},
    {"learning_rate": 0.05, "max_depth": 7, "min_child_weight": 3.0, "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 2.0},
    {"learning_rate": 0.03, "max_depth": 8, "min_child_weight": 5.0, "subsample": 0.8,  "colsample_bytree": 0.8,  "reg_lambda": 2.0},
]
results = []
for override in PARAM_GRID:
    params = {**CONFIG.baseline_params, **override}
    results.append({"override": override, "params": params, **run_cv(params, labeled_df, y_labeled)})
results.sort(key=lambda r: r["overall"])
best = results[0]
best["overall"]

In [ ]:
# === 35. Pseudo-labeling 시도 (CV log-loss 개선 시에만 채택) ===
def make_pseudo_labels(params, X_lab, y_lab, X_unl, threshold, num_boost_round):
    X_tr_enc, X_unl_enc, _ = encode_train_valid(X_lab, y_lab, X_unl)
    b = xgb.train(params, xgb.DMatrix(X_tr_enc, label=y_lab), num_boost_round)
    probs = b.predict(xgb.DMatrix(X_unl_enc))
    keep = (probs >= threshold) | (probs <= 1 - threshold)
    pseudo_y = pd.Series(np.where(probs >= 0.5, 1, 0), index=X_unl.index)
    return X_unl.loc[keep].reset_index(drop=True), pseudo_y.loc[keep].reset_index(drop=True)

def run_cv_with_pseudo(params, X_lab, y_lab, pseudo_X, pseudo_y,
                       n_splits=CONFIG.n_splits, num_boost_round=None,
                       early_stopping_rounds=None):
    num_boost_round = num_boost_round or CONFIG.baseline_num_boost_round
    early_stopping_rounds = early_stopping_rounds or CONFIG.early_stopping_rounds
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X_lab), dtype=float)
    fold_scores, best_iters = [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), 1):
        X_tr = pd.concat([X_lab.iloc[tr_idx], pseudo_X], ignore_index=True)
        y_tr = pd.concat([y_lab.iloc[tr_idx], pseudo_y], ignore_index=True)
        X_va, y_va = X_lab.iloc[va_idx].reset_index(drop=True), y_lab.iloc[va_idx].reset_index(drop=True)
        X_tr_enc, X_va_enc, _ = encode_train_valid(X_tr, y_tr, X_va)
        b = xgb.train(params, xgb.DMatrix(X_tr_enc, label=y_tr), num_boost_round,
                      evals=[(xgb.DMatrix(X_va_enc, label=y_va), "valid")],
                      early_stopping_rounds=early_stopping_rounds, verbose_eval=False)
        p = b.predict(xgb.DMatrix(X_va_enc), iteration_range=(0, b.best_iteration + 1))
        oof[va_idx] = p
        fold_scores.append(log_loss(y_va, p)); best_iters.append(b.best_iteration)
    overall = log_loss(y_lab, oof)
    return {"oof": oof, "fold_scores": fold_scores, "overall": overall,
            "mean_best_iter": int(np.mean(best_iters))}

pseudo_X, pseudo_y = make_pseudo_labels(best["params"], labeled_df, y_labeled, unlabeled_df,
                                        CONFIG.pseudo_threshold, best["mean_best_iter"])
if len(pseudo_X):
    cv_pseudo = run_cv_with_pseudo(best["params"], labeled_df, y_labeled, pseudo_X, pseudo_y)
    use_pseudo = cv_pseudo["overall"] < best["overall"]
else:
    cv_pseudo, use_pseudo = None, False
use_pseudo

In [ ]:
# === 36. 최종 학습 — best params + (선택)pseudo, test 예측 ===
if use_pseudo:
    X_final = pd.concat([labeled_df, pseudo_X], ignore_index=True)
    y_final = pd.concat([y_labeled, pseudo_y], ignore_index=True)
    num_rounds_final = cv_pseudo["mean_best_iter"]
else:
    X_final, y_final = labeled_df.copy(), y_labeled.copy()
    num_rounds_final = best["mean_best_iter"]

X_final_enc, X_test_enc, _ = encode_train_valid(X_final, y_final, test_df_X)
dfinal = xgb.DMatrix(X_final_enc, label=y_final)
dtest = xgb.DMatrix(X_test_enc)
final_booster = xgb.train(best["params"], dfinal, max(num_rounds_final, 1))
test_proba_delayed = final_booster.predict(dtest)
pd.Series(test_proba_delayed).describe()

In [ ]:
# === 37. 제출 CSV 작성 (ID, Not_Delayed, Delayed) + 합=1 검증 ===
assert len(test_ids) == len(test_proba_delayed), "ID 와 예측 수 불일치"
submission = pd.DataFrame({
    CONFIG.id_col: test_ids.values,
    CONFIG.negative_label: 1.0 - test_proba_delayed,
    CONFIG.positive_label: test_proba_delayed,
})
assert np.allclose(submission[[CONFIG.negative_label, CONFIG.positive_label]].sum(axis=1), 1.0), "합 != 1"
CONFIG.submission_dir.mkdir(parents=True, exist_ok=True)
path = CONFIG.submission_dir / CONFIG.submission_filename
submission.to_csv(path, index=False)